# Market-Based / Bidding Pattern | Multi-Agent Collaboration

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, List, Dict
from typing_extensions import NotRequired
from concurrent.futures import ThreadPoolExecutor
import json
import re
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache
from helper import plot_mermaid, stream_invoke

In [2]:
# Setup Response caching
set_llm_cache(SQLiteCache(database_path=".langchain_cache.db"))

In [3]:
model = ChatOpenAI(model="gpt-4o")

In [4]:
# Agent profiles with capabilities
AGENT_PROFILES = {
    "data_scientist": {
        "skills": ["statistics", "ML", "data analysis", "Python", "visualization"],
        "current_load": 0.3,  # 0.0 = idle, 1.0 = fully loaded
    },
    "web_developer": {
        "skills": ["HTML", "CSS", "JavaScript", "React", "APIs", "frontend"],
        "current_load": 0.1,
    },
    "technical_writer": {
        "skills": ["documentation", "tutorials", "API docs", "technical writing", "editing"],
        "current_load": 0.5,
    },
}

class MarketState(TypedDict):
    task: str
    bids: NotRequired[Dict[str, Dict]]  # Multi-dimensional bids per agent
    winner: NotRequired[str]
    result: NotRequired[str]

In [5]:
def collect_bids(state: MarketState) -> dict:
    """Each agent evaluates the task and submits a multi-dimensional bid."""
    def get_bid(agent_name: str) -> tuple:
        profile = AGENT_PROFILES[agent_name]
        response = model.invoke(
            f"You are '{agent_name}' with skills: {profile['skills']}.\n"
            f"Your current workload: {profile['current_load']*100:.0f}%.\n\n"
            f"Task: {state['task']}\n\n"
            f"Rate yourself on TWO dimensions (0.0 to 1.0 each):\n"
            f"- skill_match: How well your skills match the task requirements\n"
            f"- bid_quality: How high-quality a result you can deliver for this specific task\n\n"
            f'Return JSON: {{\"skill_match\": 0.85, \"bid_quality\": 0.7, \"reason\": \"brief explanation\"}}'
        )
        cleaned = re.sub(r"```(?:json)?\s*|\s*```", "", response.content).strip()
        try:
            parsed = json.loads(cleaned)
            skill_match = float(parsed.get("skill_match", 0.5))
            bid_quality = float(parsed.get("bid_quality", 0.5))
        except (json.JSONDecodeError, ValueError):
            skill_match, bid_quality = 0.5, 0.5
        availability = round(1.0 - profile["current_load"], 2)
        return (agent_name, {"skill_match": round(skill_match, 2),
                             "availability": availability,
                             "bid_quality": round(bid_quality, 2)})

    with ThreadPoolExecutor() as executor:
        bid_results = list(executor.map(get_bid, AGENT_PROFILES.keys()))
    bids = dict(bid_results)
    return {"bids": bids}

def select_winner(state: MarketState) -> dict:
    """Score bids using weighted multi-dimensional evaluation."""
    bids = state.get("bids", {})
    if not bids:
        return {"winner": list(AGENT_PROFILES.keys())[0]}
    # Weighted scoring: skill_match (50%) + availability (30%) + bid_quality (20%)
    scores = {}
    for agent, dims in bids.items():
        total = dims["skill_match"] * 0.5 + dims["availability"] * 0.3 + dims["bid_quality"] * 0.2
        scores[agent] = round(total, 3)
        print(f"  {agent}: skill={dims['skill_match']}, availability={dims['availability']}, "
              f"quality={dims['bid_quality']}, total={scores[agent]}")
    winner = max(scores, key=scores.get)
    print(f"  -> Winner: {winner}")
    return {"winner": winner}

def execute_task(state: MarketState) -> dict:
    """The winning agent executes the task."""
    winner = state["winner"]
    profile = AGENT_PROFILES[winner]
    response = model.invoke(
        f"You are '{winner}' with skills in {profile['skills']}.\n"
        f"You won the bid to complete this task.\n\n"
        f"Task: {state['task']}\n\n"
        f"Complete the task thoroughly using your expertise."
    )
    bid_summary = ", ".join(f"{k}: {v}" for k, v in state.get("bids", {}).items())
    return {"result": f"Winner: {winner} (Bids: {bid_summary})\n\n{response.content}"}

In [6]:
graph = StateGraph(MarketState)
graph.add_sequence([("collect_bids", collect_bids), ("select_winner", select_winner), ("execute_task", execute_task)])
graph.add_edge(START, "collect_bids")
graph.add_edge("execute_task", END)

market = graph.compile()

In [7]:
# Plot the workflow
plot_mermaid(market)

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	collect_bids(collect_bids)
	select_winner(select_winner)
	execute_task(execute_task)
	__end__([<p>__end__</p>]):::last
	__start__ --> collect_bids;
	collect_bids --> select_winner;
	select_winner --> execute_task;
	execute_task --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
result = market.invoke({"task": "Create a data visualization dashboard showing monthly sales trends with interactive filters"})
print(result["result"])

  data_scientist: skill=0.85, availability=0.7, quality=0.7, total=0.775
  web_developer: skill=0.85, availability=0.9, quality=0.7, total=0.835
  technical_writer: skill=0.2, availability=0.5, quality=0.4, total=0.33
  -> Winner: web_developer
Winner: web_developer (Bids: data_scientist: {'skill_match': 0.85, 'availability': 0.7, 'bid_quality': 0.7}, web_developer: {'skill_match': 0.85, 'availability': 0.9, 'bid_quality': 0.7}, technical_writer: {'skill_match': 0.2, 'availability': 0.5, 'bid_quality': 0.4})

To create an interactive data visualization dashboard for showing monthly sales trends, I'll use my web development expertise, focusing on frontend technologies along with data handling. The key goal is to develop a user-friendly interface that allows users to interactively explore sales data. Here’s a step-by-step approach to completing the task:

### 1. Define Project Requirements

- **Data Source**: We'll assume there's an API available providing monthly sales data, which inclu

In [9]:
stream_invoke(market, {"task": "Create a data visualization dashboard showing monthly sales trends with interactive filters"})


────────────────────────────────────────────────────────────────────────────────
  STREAMING EXECUTION
────────────────────────────────────────────────────────────────────────────────

  data_scientist: skill=0.85, availability=0.7, quality=0.7, total=0.775
  web_developer: skill=0.85, availability=0.9, quality=0.7, total=0.835
  technical_writer: skill=0.2, availability=0.5, quality=0.4, total=0.33
  -> Winner: web_developer
────────────────────────────────────────────────────────────────────────────────
  EXECUTION COMPLETE
────────────────────────────────────────────────────────────────────────────────



{'task': 'Create a data visualization dashboard showing monthly sales trends with interactive filters',
 'bids': {'data_scientist': {'skill_match': 0.85,
   'availability': 0.7,
   'bid_quality': 0.7},
  'web_developer': {'skill_match': 0.85,
   'availability': 0.9,
   'bid_quality': 0.7},
  'technical_writer': {'skill_match': 0.2,
   'availability': 0.5,
   'bid_quality': 0.4}},
 'winner': 'web_developer',
 'result': 'Winner: web_developer (Bids: data_scientist: {\'skill_match\': 0.85, \'availability\': 0.7, \'bid_quality\': 0.7}, web_developer: {\'skill_match\': 0.85, \'availability\': 0.9, \'bid_quality\': 0.7}, technical_writer: {\'skill_match\': 0.2, \'availability\': 0.5, \'bid_quality\': 0.4})\n\nTo create an interactive data visualization dashboard for showing monthly sales trends, I\'ll use my web development expertise, focusing on frontend technologies along with data handling. The key goal is to develop a user-friendly interface that allows users to interactively explore sal